# 04 Record Rollout

## Objective
Record real observations through RolloutRecorder and export JSON, CSV, NPZ, metadata, and a manifest.

## Prerequisites
FlyGym 2.1.0, MuJoCo 3.9.0, and the adapter package.

## Expected Output
A bounded preview package under results/colab/rollout_preview.

## Troubleshooting
If COM or contact is unavailable, inspect the metadata and sensor configuration before interpreting the result.

## Validation
The export manifest and printed file list provide a bounded rollout integrity check.

## Next notebook
05_Generate_Healthy_001.ipynb

In [ ]:
import os
from pathlib import Path
repo = Path.cwd() / 'drosophila-pd-flygym'
if (repo / 'pyproject.toml').is_file():
    os.chdir(repo)
try:
    from drosophila_pd.flygym_adapter import FlyBuilder, FlyGymConfig, FlyGymRuntime, RolloutRecorder, SimulationBuilder, WorldBuilder, export_rollout
    config = FlyGymConfig.from_yaml('configs/v2/flygym/default.yaml')
    fly = FlyBuilder().healthy().build()
    world = WorldBuilder().flat().with_fly(fly, position=config.world.spawn_position, orientation=config.world.spawn_orientation, add_ground_contact_sensors=config.world.add_ground_contact_sensors).build()
    simulation = SimulationBuilder().with_world(world).timestep(config.simulation.timestep).build()
    simulation.reset()
    recorder = RolloutRecorder(simulation, fly.name, fly=fly, timestep=simulation.mj_model.opt.timestep, simulation_metadata=config.to_mapping())
    runtime = FlyGymRuntime(simulation, recorder=recorder, max_steps=5)
    runtime.run()
    exported = export_rollout(recorder.rollout, 'results/colab/rollout_preview')
    print('frames:', recorder.rollout.frame_count)
    print('files:', exported.files)
except Exception as exc:
    print('Rollout recording failed:', type(exc).__name__, exc)